In [13]:
import os
import numpy as np
import pandas as pd
from project_manager import read_json_to_dict
import scipy.stats as stats
import matplotlib.pyplot as plt

In [15]:
%matplotlib inline

In [2]:
project_dir = r'C:\Users\10007166\dbh_autoeda\AutoEDA Sample'
data_path = os.path.join(project_dir, 'data.csv')
column_info_path = os.path.join(project_dir, 'column_info.json')
df = pd.read_csv(data_path)
col_info = read_json_to_dict(column_info_path)

In [33]:
[i for i,info in enumerate(col_info) if info['ColumnChar']=='Continuous']

[5, 7, 8, 13]

In [34]:
idx = 13
info = col_info[idx]
data = df[info['Column']]
print(info['Column'], info['ColumnChar'])

column14 Continuous


정규성 검정

목적 : 데이터가 정규 분포를 따르는지 확인

1. 가설 설정
- 귀무가설 (H0): 데이터가 정규 분포를 따른다.
- 대립가설 (H1): 데이터가 정규 분포를 따르지 않는다.

2. p-value와 유의수준 비교
- p-value : 귀무가설이 참일 때, 극단적인 데이터를 얻을 확률. 즉 p-value가 작을수록 귀무가설이 참일 가능성이 낮다는 것을 의미
- 유의수준(α) : 일반적으로 0.05, 0.01을 사용. 귀무가설을 기각할 기준이 되는 값. 5%의 확률로 귀무가설을 잘못 기각할 수 있다는 의미

3. 결정 규칙
- p-value <= α : 귀무가설 기각. 즉 데이터는 정규 분포를 따르지 않는다고 결론
- p-value > α : 귀무가설 기각 X. 즉, 데이터는 정규 분포를 따른다고 결론

검정 기법
- Shapiro-Wilk Test : 소표본에 적합, n<50 일 때 강력
- Kolmogorov-Smirnov Test : 정밀한 정규성 검정 필요 시 적합, 분포의 꼬리부분까지 민감하게 검정
- Anderson-Darling Test : 일반적인 확인용

In [ ]:
# 정규성 검정

# 목적 : 데이터가 정규 분포를 따르는지 확인

# 1. 가설 설정


In [126]:
info = col_info[7]
col_nm = info['Column']
data = df[col_nm]
mean, std = info['Mean'], info['Std']
print(col_nm, len(data), mean, std)

column8 500 2.020288931323474 2.0127595039858126


In [128]:
from scipy.stats import shapiro, kstest, anderson

In [137]:
f'{1.2:.1f}'

'1.2'

In [141]:
def test_result_by_p_value(p_value, significance_level, col_nm):
    if p_value <= significance_level:
        return {
            'Result': f'{col_nm} 은 정규 분포를 따르지 않는다.',
            'Detail': f'p-value ({p_value:.4f}) ≤ α ({significance_level:.2f}) 이므로, 귀무가설(H0) 기각'
        }
    return {
        'Result': f'{col_nm} 은 정규 분포를 따른다.',
        'Detail': f'p-value ({p_value:.4f}) > α ({significance_level:.2f}) 이므로, 귀무가설(H0) 채택'
    }

def test_result_by_stat(stat, threshold, significance_level, col_nm):
    if stat < threshold:
        return {
            'Result': f'{col_nm} 은 정규 분포를 따른다.',
            'Detail': f'통계량 ({stat:.4f}) < 임계값 ({threshold:.4f}, 유의수준 {significance_level:.1f}%) 이므로, 귀무가설(H0) 채택'
        }
    return {
        'Result': f'{col_nm} 은 정규 분포를 따르지 않는다.',
        'Detail': f'p-통계량 ({stat:.4f}) ≥ 임계값 ({threshold:.4f}, 유의수준 {significance_level:.1f}) 이므로, 귀무가설(H0) 기각'
    }

In [142]:
target = np.array(data)

stat, p = shapiro(target)
print(f'Shapiro-Wilk 통계량 = {stat:.4f}')
print(f'p-value = {p:.4f}')

result = {
    'Method': 'Shapiro Wilk',
    'BestCase': '샘플 크기가 작거나 중간 규모인 경우',
    'Strength': '작은 표본에서 높은 검정력',
    'Weakness': '큰 데이터셋에서는 매우 민감',
    '0.01': test_result_by_p_value(p, 0.01, col_nm),
    '0.05': test_result_by_p_value(p, 0.05, col_nm)
}

result

Shapiro-Wilk 통계량 = 0.8209
p-value = 0.0000


{'Method': 'Shapiro Wilk',
 'BestCase': '샘플 크기가 작거나 중간 규모인 경우',
 'Strength': '작은 표본에서 높은 검정력',
 'Weakness': '큰 데이터셋에서는 매우 민감',
 '0.01': {'Result': 'column8 은 정규 분포를 따르지 않는다.',
  'Detail': 'p-value (0.0000) ≤ α (0.01) 이므로, 귀무가설(H0) 기각'},
 '0.05': {'Result': 'column8 은 정규 분포를 따르지 않는다.',
  'Detail': 'p-value (0.0000) ≤ α (0.05) 이므로, 귀무가설(H0) 기각'}}

In [143]:
stat, p = kstest(target, 'norm', args=(mean, std))
print(f'K-S 통계량 = {stat:.4f}')
print(f'p-value = {p:.4f}')

result = {
    'Method': 'Kolmogorov-Smirnov',
    'BestCase': '분포 전체를 단수하게 비교할 때, 평균/표준편차가 이미 알려진 경우',
    'Strength': '표본 크기에 제한 없음',
    'Weakness': '검정 정밀도 낮음',
    '0.01': test_result_by_p_value(p, 0.01, col_nm),
    '0.05': test_result_by_p_value(p, 0.05, col_nm)
}

result

K-S 통계량 = 0.1581
p-value = 0.0000


{'Method': 'Kolmogorov-Smirnov',
 'BestCase': '분포 전체를 단수하게 비교할 때, 평균/표준편차가 이미 알려진 경우',
 'Strength': '표본 크기에 제한 없음',
 'Weakness': '검정 정밀도 낮음',
 '0.01': {'Result': 'column8 은 정규 분포를 따르지 않는다.',
  'Detail': 'p-value (0.0000) ≤ α (0.01) 이므로, 귀무가설(H0) 기각'},
 '0.05': {'Result': 'column8 은 정규 분포를 따르지 않는다.',
  'Detail': 'p-value (0.0000) ≤ α (0.05) 이므로, 귀무가설(H0) 기각'}}

In [146]:
stat = anderson(target, dist='norm')

print(f'검정 통계치: {stat.statistic}')
print(f'임계지점의 통계치: {stat.critical_values}')
print(f'각 임계 지점의 p-value: {stat.significance_level}')

result = {
    'Method': 'Anderson-Darling',
    'BestCase': '꼬리 부분이 중요한 경우',
    'Strength': '정규성뿐 아니라 꼬리 적합성까지 민감하게 포착',
    'Weakness': '표본이 적으면 불안정',
    '0.01': test_result_by_stat(stat.statistic, stat.critical_values[-1], stat.significance_level[-1], col_nm),
    '0.05': test_result_by_stat(stat.statistic, stat.critical_values[2], stat.significance_level[2], col_nm)
}
result

검정 통계치: 26.267206394053346
임계지점의 통계치: [0.571 0.651 0.781 0.911 1.083]
각 임계 지점의 p-value: [15.  10.   5.   2.5  1. ]


{'Method': 'Anderson-Darling',
 'BestCase': '꼬리 부분이 중요한 경우',
 'Strength': '정규성뿐 아니라 꼬리 적합성까지 민감하게 포착',
 'Weakness': '표본이 적으면 불안정',
 '0.01': {'Result': 'column8 은 정규 분포를 따르지 않는다.',
  'Detail': 'p-통계량 (26.2672) ≥ 임계값 (1.0830, 유의수준 1.0) 이므로, 귀무가설(H0) 기각'},
 '0.05': {'Result': 'column8 은 정규 분포를 따르지 않는다.',
  'Detail': 'p-통계량 (26.2672) ≥ 임계값 (0.7810, 유의수준 5.0) 이므로, 귀무가설(H0) 기각'}}